In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
doc = documents[0]

In [8]:
import json
user_prompt = json.dumps(doc)

In [9]:
user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['What does RAG actually do to help an LLM answer questions better?', 'Why does the course treat the language model like a black box instead of explaining how it works inside?', 'What are the main limits of LLMs that RAG is meant to fix?', 'What kind of example project will this module build to show RAG in practice?', 'What topics are covered in the first part of the module, and how is the second part different?']


In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What is the main idea behind retrieval-augmented generation, and how does it help with model limits?', 'Why does the course treat the language model like a black box instead of going into how it works internally?', 'What kinds of problems can happen if you rely on a model’s built-in knowledge only?', 'What is the FAQ assistant project in this module supposed to do?', 'What will be covered in the first part of the module versus the second part?']


In [15]:
usage.input_tokens, usage.output_tokens

(1020, 105)

In [17]:
for doc in documents[:3]:
    user_prompt = json.dumps(doc)
    messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}    
    ]
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    print(result.questions)
    print(usage.input_tokens, usage.output_tokens)

['What is retrieval-augmented generation, and why does it help with the limits of an LLM?', 'Why do you use a model as a black box in this course instead of training or hosting one yourself?', 'What kinds of problems can happen if you rely on an LLM without giving it extra context?', 'What is the FAQ agent in this module supposed to do, and what data does it answer from?', 'What will be built in the first part of the module, and how is the second part different?']
1020 117
['What do I need installed before starting this module, and do I need anything besides Python and Jupyter?', 'How do I set up a brand-new project for this course using uv, from creating the folder to adding the needed packages?', 'What’s the safest way to keep my API key for OpenAI or a similar provider so I don’t commit it by accident?', 'How do I launch Jupyter for the course and test that the OpenAI client is working in a notebook?', 'If I’m using Groq or another OpenAI-style API, how do I configure the client dif

In [7]:
import pandas as pd
df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth) 

360

In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [9]:
from minsearch import Index, VectorSearch
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [10]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [11]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [12]:
text_results = text_search(q)
text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [13]:
from embedder import Embedder

embed = Embedder()

In [14]:
texts = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(texts)

In [15]:
X.shape

(295, 384)

In [16]:
vector_index = VectorSearch()
vector_index.fit(X, chunks)

In [17]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

In [18]:
vector_results = vector_search(q)
vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [19]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [20]:
def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

In [21]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance


In [22]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total

In [23]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

In [24]:
def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

In [25]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [26]:
result_text = evaluate(ground_truth, text_search)
print(result_text)
result_vector = evaluate(ground_truth, vector_search)
print(result_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


In [27]:
results_by_k = {}
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    results_by_k[k] = result
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
